# Update Gronings↔Dutch Translation Model

This notebook walks through the full pipeline for updating the
[HuggingFace model](https://huggingface.co/Tom9358/nllb-tatoeba-gos-nld-v1):

1. **Download** fresh Tatoeba data
2. **Train** with the best settings (focused multilingual preset, 12 epochs)
3. **Evaluate** all checkpoints
4. **Inspect** sample translations
5. **Upload** to HuggingFace

Run cells top-to-bottom. You only need to set the Huggingface token in .env

## 1. Configuration

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

from nllb_try.config import RunConfig
from nllb_try.corpus import main_corpus, pool_varieties_into_tatoeba
from nllb_try.downloadtatoeba import main_download

DATA_DIR = "data"

cfg = RunConfig(
    modelname="facebook/nllb-200-distilled-600M",
    source_langs_tatoeba=("nld", "gos", "deu", "eng", "spa"),
    source_langs_nllb=(
        "nld_Latn",
        "gos_Latn",
        "deu_Latn",
        "eng_Latn",
        "spa_Latn",
    ),
    new_lang_nllb="gos_Latn",
    similar_lang_nllb="nld_Latn",
    tatoeba_path=os.path.join(DATA_DIR, "tatoeba"),
    data_root_path=DATA_DIR,
    model_cache_path="hfacemodels",
    batch_size=256,
    num_epochs=12,
    warmup_steps=70,
    sampling_temperature=5.0,
    sampling_strategy="focus_cap",
    focus_lang_pair=("nld_Latn", "gos_Latn"),
    max_length=47,
)
print(f"Run ID: {cfg.run_id}")
print(f"Run dir: {cfg.run_dir}")

## 2. Download & Build Corpora

Downloads fresh Tatoeba data, builds all language pair corpora with the
global sentence-ID split, loads variety data, pools variety training
rows into their matching Tatoeba corpus, and trains with focused
multilingual sampling.

In [ ]:
VARIETY_DIR = os.path.join(DATA_DIR, "RUG_data", "csv")

# Download fresh Tatoeba data
main_download(cfg.source_langs_tatoeba, redownload=True, tatoeba_path=cfg.tatoeba_path)

# Build all corpora (Tatoeba pairs + variety CSVs)
all_corpora = main_corpus(
    source_langs_tatoeba=cfg.source_langs_tatoeba,
    source_langs_nllb=cfg.source_langs_nllb,
    variety_dir=None, #VARIETY_DIR
    cfg=cfg,
)

# Pool variety training data into Tatoeba corpora for training
train_corpora = pool_varieties_into_tatoeba(all_corpora)

print("\n--- Training corpora (focused multilingual) ---")
total = 0
for c in train_corpora:
    print(
        f"  {c.source_lang_nllb}-{c.target_lang_nllb}: "
        f"train={len(c.df_train):,}, val={len(c.df_validate):,}"
    )
    total += len(c.df_train)
print(f"  Total: {total:,} training pairs")

## 3. Train

In [ ]:
from nllb_try.train import main_train

main_train(train_corpora, cfg)
print(f"Training complete. Run dir: {cfg.run_dir}")

## 4. Evaluate

Evaluates all epoch checkpoints on all corpora (Tatoeba + variety),
including the untrained baseline for reference. Uses `all_corpora`
(unpooled) so per-variety metrics are reported separately.

In [ ]:
from nllb_try.evaluate import main_evaluate

main_evaluate(
    corpus_objects=all_corpora,
    run_dir=cfg.run_dir,
    new_lang_nllb=cfg.new_lang_nllb,
    device=cfg.device,
    sample_size=750,
    include_baseline=True,
    verbose=True,
)
print("Evaluation complete — check the eval/ subfolder for metrics.csv and plots.")

## 5. Inspect Translations

Load the best epoch and try some sample translations.
Adjust `EPOCH` below based on the evaluation results above.

In [ ]:
import json
import os

from nllb_try.evaluate import translate
from nllb_try.tokenizer_and_model_setup import setup_model_and_tokenizer

USE_EXISTING_RUN = True
EPOCH = "epoch12"

if USE_EXISTING_RUN:
    run_dir = "checkpoints/nllb-200-distilled-600M-nld-gos-deu-eng-spa-20260725-203524"
    with open(os.path.join(run_dir, "run_config.json"), encoding="utf-8") as f:
        run_cfg = json.load(f)

    model_cache_path = run_cfg.get("model_cache_path", "hfacemodels")
    new_lang_nllb = run_cfg.get("new_lang_nllb", "gos_Latn")
    device = run_cfg.get("device", "cuda")
else:
    run_dir = cfg.run_dir
    model_cache_path = cfg.model_cache_path
    new_lang_nllb = cfg.new_lang_nllb
    device = cfg.device

model_path = os.path.join(run_dir, "checkpoints", EPOCH)
print(f"Loading {model_path}...")

model, tokenizer = setup_model_and_tokenizer(
    model_path,
    modelpath=model_cache_path,
    new_lang=new_lang_nllb,
    device=device,
)

# Sample sentences
sentences_nld = [
    "Ik ga morgen naar de stad.",
    "Het regent buiten. Neem een paraplu mee!",
    "De kat slaapt zachtjes op de bank.",
    "Wij drinken koffie bij oma.",
    "Hoe gaat het met jou?",
    "Die film was echt heel spannend.",
    "Het kind speelt in de tuin.",
    "Ik weet niet wat je bedoelt.",
    "De man die je gisteren zag, is mijn buurman.",
    "Het kind dat in de speeltuin speelt, is mijn neefje.",
]

print("\n--- Nederlands → Gronings ---")
translations_gos = []
for s in sentences_nld:
    t = translate(
        s, src_lang="nld_Latn", tgt_lang="gos_Latn", model=model, tokenizer=tokenizer
    )
    translations_gos.append(t)
    print(f"  NL:  {s}")
    print(f"  GOS: {t}\n")

print("--- Gronings → Nederlands (back-translation) ---")
for gos in translations_gos:
    back = translate(
        gos, src_lang="gos_Latn", tgt_lang="nld_Latn", model=model, tokenizer=tokenizer
    )
    print(f"  GOS: {gos}")
    print(f"  NL:  {back}\n")

In [ ]:
import huggingface_hub

from nllb_try.evaluate import translate
from nllb_try.tokenizer_and_model_setup import setup_model_and_tokenizer

REPO_ID = "Tom9358/nllb-tatoeba-gos-nld-v1"
EPOCH = "epoch12"

huggingface_hub.login(token=os.environ["HF_TOKEN"])

# print(tokenizer.convert_tokens_to_ids("gos_Latn"))

# model_path = os.path.join(cfg.run_dir, "checkpoints", EPOCH)
# print(f"Loading {model_path}...")
# model, tokenizer = setup_model_and_tokenizer(
#     model_path,
#     modelpath=cfg.model_cache_path,
#     new_lang=cfg.new_lang_nllb,
#     device=cfg.device,
# )

# print(translate(
#     "Het tuinhuis van suikerruiker.",
#     src_lang="nld_Latn",
#     tgt_lang="gos_Latn",
#     model=model,
#     tokenizer=tokenizer,
# ))
tokenizer.push_to_hub(REPO_ID)
model.push_to_hub(REPO_ID)

## 6. Upload to HuggingFace

Log in and push the model + tokenizer. Set `HF_TOKEN` in `.env`; get a
write-enabled token from https://huggingface.co/settings/tokens.

In [ ]:
import os

import huggingface_hub
from dotenv import load_dotenv

load_dotenv()
huggingface_hub.login(token=os.environ["HF_TOKEN"])

In [ ]:
REPO_ID = "Tom9358/nllb-tatoeba-gos-nld-v1"

tokenizer.push_to_hub(REPO_ID)
model.push_to_hub(REPO_ID)